In [1]:
import numpy as np
import matplotlib.pyplot as plt

from pydrake.all import (
    AddMultibodyPlant,
    ConstantVectorSource,
    Cylinder,
    CoulombFriction,
    DiagramBuilder,
    Meshcat,
    MeshcatVisualizer,
    MeshcatVisualizerParams,
    MultibodyPlantConfig,
    Parser,
    PdControllerGains,
    PidController,
    RigidTransform,
    Simulator,
    VectorLogSink,
    SpatialInertia,
)

In [2]:
meshcat = Meshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [3]:
builder = DiagramBuilder()

plant_config = MultibodyPlantConfig()
plant_config.time_step = 1e-3

plant, scene_graph = AddMultibodyPlant(plant_config, builder)

panda_model_instance = Parser(plant, scene_graph).AddModelsFromUrl(
    "package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
plant.WeldFrames(
    plant.world_frame(),
    plant.GetRigidBodyByName("panda_link0").body_frame(),
    RigidTransform()
)
plant.set_gravity_enabled(panda_model_instance, False)

# Add end effector
tool_model_instance = plant.AddModelInstance("tool")
flange = plant.AddRigidBody(
    "flange", tool_model_instance,
    SpatialInertia.SolidCylinderWithDensity(1000, 0.05, 0.02, [0,0,1]))
plant.WeldFrames(
    plant.GetRigidBodyByName("panda_link7").body_frame(),
    flange.body_frame(), RigidTransform([0, 0, 0.11]))
plant.RegisterVisualGeometry(flange, RigidTransform(), Cylinder(0.05, 0.02), "flange", [0, 0, 1, 1])
plant.RegisterCollisionGeometry(flange, RigidTransform(), Cylinder(0.05, 0.02), "flange", CoulombFriction(0.1,0.1))

plant.Finalize()

q0 = [0, -np.pi*0.2, 0, -np.pi*0.9, 0, np.pi*0.7, 0]
plant.SetDefaultPositions(q0)


# Add Meshcat visualizer
visualizer = MeshcatVisualizer.AddToBuilder(
    builder, scene_graph, meshcat, MeshcatVisualizerParams()
)

# Add PID controller
Kp = 400
Ki = 0
Kd = 20
pid = builder.AddSystem(PidController(
    np.ones(7) * Kp, np.ones(7) * Ki, np.ones(7) * Kd))

# Add zero state input for panda
source = builder.AddSystem(ConstantVectorSource(
    np.concatenate((q0, np.zeros(7)))))
builder.Connect(
    source.get_output_port(),
    pid.get_input_port_desired_state()
)
builder.Connect(
    plant.get_state_output_port(),
    pid.get_input_port_estimated_state()
)
builder.Connect(
    pid.get_output_port(),
    plant.get_actuation_input_port()
)

# Build and simulate
diagram = builder.Build()
simulator = Simulator(diagram)
context = simulator.get_mutable_context()
plant_context = diagram.GetSubsystemContext(plant, context)

simulator.Initialize()
simulator.set_target_realtime_rate(1.0)

simulator.AdvanceTo(2.0)